# Recogida sin ruido

Aquest notebook fa la mateixa idea que `dades.ipynb`, pero amb una diferencia important: les imatges son netes, sense pixels propers ni pixels aleatoris.

La mostra de colors es exactament la mateixa que ja tenim a `csv/input_image_sample.csv`. Aixi podem comparar despres si el soroll de la imatge afecta els models.


## Configuracio

Definim les rutes separades. Tot el que generi aquest notebook va a `experiments/sense-soroll/`, per no barrejar-ho amb l'experiment normal.

In [ ]:
from pathlib import Path
import sys
import time

IMAGE_SIZE = 100

# Aquesta part crida l'API i pot gastar diners. Ho deixo apagat per defecte.
RUN_MODEL_QUERIES = True
MODEL_NAMES = ["gpt-4o", "gpt-4o-mini"]
MODEL_TEMPERATURE = 0.2
MODEL_MAX_IMAGES = None  # posa un numero petit, per exemple 5, si vols fer una prova rapida
RETRY_FAILED_MODEL_QUERIES = True

# Canvia a True nomes si vols tornar a crear les imatges netes des de zero.
REGENERATE_IMAGES = False

# Casella de seguretat: esborra nomes fitxers de sense-soroll, no toca l'experiment original.
DELETE_EXISTING_SAMPLE = False


def _format_duration_local(seconds):
    seconds = max(0, float(seconds))
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, remaining_seconds = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {remaining_seconds:04.1f}s"
    hours, remaining_minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(remaining_minutes)}m {remaining_seconds:04.1f}s"


def _install_cell_timer():
    ipython = get_ipython()
    if ipython is None or getattr(ipython, "_color_llm_timer_installed", False):
        return

    def _start_cell_timer(info):
        ipython.user_ns["_cell_started_at"] = time.perf_counter()

    def _stop_cell_timer(result):
        started_at = ipython.user_ns.get("_cell_started_at")
        if started_at is not None:
            elapsed = time.perf_counter() - started_at
            print(f"Temps cel.la: {_format_duration_local(elapsed)}")

    ipython.events.register("pre_run_cell", _start_cell_timer)
    ipython.events.register("post_run_cell", _stop_cell_timer)
    ipython._color_llm_timer_installed = True


_install_cell_timer()

base_dir = Path("recollida-dades")
if not base_dir.exists():
    base_dir = Path("..").resolve()
if not (base_dir / "scripts").exists():
    raise FileNotFoundError(f"No trobo recollida-dades/scripts des de {Path.cwd()}")
source_csv_dir = base_dir / "experiments" / "soroll" / "csv"
source_input_path = source_csv_dir / "input_image_sample.csv"

experiment_dir = base_dir / "experiments" / "sense-soroll"
csv_dir = experiment_dir / "csv"
images_dir = experiment_dir / "images"
tmp_dir = base_dir / "archive" / "tmp" / "sense-soroll-notebook"
logs_dir = experiment_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, tmp_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)
print("Carpeta sense soroll:", experiment_dir)

## Carrega de scripts

Importem les funcions comunes. Aqui necessitem especialment la funcio que genera imatges 100% del color objectiu.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display
from PIL import Image

import utils
importlib.reload(utils)

build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
delete_sample_outputs = utils.delete_sample_outputs
delete_png_files = utils.delete_png_files
generate_solid_images_from_sample = utils.generate_solid_images_from_sample
save_csv = utils.save_csv
save_rgb_sample_grid = utils.save_rgb_sample_grid
save_rgb_sample_map = utils.save_rgb_sample_map
save_chroma_distribution = utils.save_chroma_distribution
save_error_distribution = utils.save_error_distribution
write_log = utils.write_log

required_functions = [
    "generate_solid_images_from_sample",
    "build_final_sample",
    "collect_model_outputs",
    "save_error_distribution",
    "delete_sample_outputs",
    "delete_png_files",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no carregats correctament: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers sense-soroll esborrats: {len(removed)}")

write_log("Notebook sense soroll inicialitzat", logs_dir / "pipeline.log")

## Carrega de la mateixa mostra de colors

No generem colors nous. Llegim la mostra original i en guardem una copia dins `experiments/sense-soroll/csv/` per tenir l'experiment net i separat.

In [ ]:
if not source_input_path.exists():
    raise FileNotFoundError(f"No trobo la mostra original: {source_input_path}")

input_sample = pd.read_csv(source_input_path)
local_input_path = csv_dir / "input_image_sample.csv"
save_csv(input_sample, local_input_path)
write_log(f"Mostra original copiada a sense-soroll: {local_input_path}", logs_dir / "pipeline.log")

print(f"Mostra carregada: {source_input_path}")
print(f"Copia local guardada: {local_input_path}")
print(f"Files: {len(input_sample)}")
display(input_sample.head())

## Mapa de color de control

Aquests grafics son els mateixos que a la mostra normal. Serveixen per comprovar que realment estem treballant amb els mateixos colors.

In [ ]:
color_grid_path = tmp_dir / "rgb_sample_grid.png"
color_map_path = tmp_dir / "rgb_sample_map.png"
chroma_plot_path = tmp_dir / "rgb_chroma_distribution.png"

save_rgb_sample_grid(input_sample, color_grid_path)
save_rgb_sample_map(input_sample, color_map_path)
save_chroma_distribution(input_sample, chroma_plot_path)
write_log(f"Visualitzacions sense-soroll generades a tmp: {color_grid_path}, {color_map_path}, {chroma_plot_path}", logs_dir / "pipeline.log")

print("Graella de les 1000 mostres:")
display(DisplayImage(filename=str(color_grid_path)))

print("Mapa de diagnosi de distribucio:")
display(DisplayImage(filename=str(color_map_path)))

print("Distribucio del chroma:")
display(DisplayImage(filename=str(chroma_plot_path)))

## Generacio de les imatges sense soroll

Aqui generem les imatges netes. Cada PNG te tots els pixels exactament iguals al color de la fila corresponent.

In [ ]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png")}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_IMAGES:
    removed_images = delete_png_files(images_dir)
    if removed_images:
        print(f"Imatges antigues sense-soroll esborrades: {len(removed_images)}")

    image_paths = generate_solid_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        progress=True,
        log_file=logs_dir / "pipeline.log",
    )
    write_log(f"Imatges sense-soroll generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges sense-soroll generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    print(f"Imatges sense-soroll ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]

## Validacio rapida de les imatges

Fem una comprovacio simple: agafem la primera imatge i mirem si tots els pixels son del color esperat.

In [ ]:
first_row = input_sample.iloc[0]
first_image_path = images_dir / first_row["image_name"]
expected_rgb = (int(first_row.r), int(first_row.g), int(first_row.b))

with Image.open(first_image_path) as image:
    pixels = list(image.getdata())

unique_pixels = set(pixels)
print("Imatge revisada:", first_image_path)
print("Color esperat:", expected_rgb)
print("Colors diferents dins la imatge:", len(unique_pixels))

if unique_pixels != {expected_rgb}:
    raise RuntimeError("La imatge no es 100% del color esperat. Alguna cosa no quadra.")

print("Validacio correcta: la imatge es completament uniforme.")

## Consulta als models

Aquesta part usa les imatges sense soroll i desa resultats nomes dins `experiments/sense-soroll/csv/`. Per defecte esta apagada per evitar crides a l'API sense voler.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

utils = importlib.reload(utils)
collect_model_outputs = utils.collect_model_outputs

load_dotenv()

output_path = csv_dir / "outputmodel_image_sample_4o.csv"
model_log_path = logs_dir / "model_calls.log"

if RUN_MODEL_QUERIES:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY. Crea un fitxer .env local amb la clau abans d'executar models.")

    client = OpenAI()
    model_outputs = collect_model_outputs(
        client=client,
        image_paths=image_paths,
        models=MODEL_NAMES,
        temperature=MODEL_TEMPERATURE,
        output_path=output_path,
        log_file=model_log_path,
        max_images=MODEL_MAX_IMAGES,
        retry_failed=RETRY_FAILED_MODEL_QUERIES,
    )

    print(f"Resultats de models sense-soroll guardats: {output_path}")
    print(f"Files totals: {len(model_outputs)}")
    if "status" in model_outputs.columns:
        print("Resum status:")
        print(model_outputs["status"].value_counts(dropna=False))
    display(model_outputs.tail())
else:
    if output_path.exists():
        model_outputs = pd.read_csv(output_path)
        print(f"Models no executats ara. Carrego resultats sense-soroll existents: {len(model_outputs)} files")
        if "status" in model_outputs.columns:
            print("Resum status:")
            print(model_outputs["status"].value_counts(dropna=False))
        display(model_outputs.tail())
    else:
        model_outputs = pd.DataFrame()
        print("Models no executats. Posa RUN_MODEL_QUERIES = True quan vulguis cridar l'API.")

## Dataset final per analisi

Quan hi hagi sortida dels models, unim entrada i resposta igual que al notebook normal, pero escrivint el resultat final a `experiments/sense-soroll/csv/sample-colors_4o.csv`.

In [ ]:
final_path = csv_dir / "sample-colors_4o.csv"
error_plot_path = tmp_dir / "error_cromatic_distribution.png"

if output_path.exists():
    model_outputs = pd.read_csv(output_path)
    final_sample = build_final_sample(input_sample, model_outputs, images_dir=images_dir)
    save_csv(final_sample, final_path)
    write_log(f"Dataset final sense-soroll guardat: {final_path} files={len(final_sample)}", logs_dir / "pipeline.log")

    print(f"Dataset final sense-soroll guardat: {final_path}")
    print(f"Files totals: {len(final_sample)}")

    print("Resum per model i status:")
    display(final_sample.groupby(["model", "status"], dropna=False).size().reset_index(name="files"))

    valid_errors = final_sample[(final_sample["status"] == "ok") & final_sample["error_cromatic"].notna()]
    if valid_errors.empty:
        print("Encara no hi ha prediccions valides per calcular l'error cromatic.")
    else:
        print("Resum error cromatic per model:")
        display(
            valid_errors.groupby("model")["error_cromatic"]
            .agg(["count", "mean", "median", "min", "max"])
            .reset_index()
        )

        save_error_distribution(final_sample, error_plot_path)
        write_log(f"Grafic error cromatic sense-soroll generat: {error_plot_path}", logs_dir / "pipeline.log")
        print("Distribucio de l'error cromatic:")
        display(DisplayImage(filename=str(error_plot_path)))

    display(final_sample.head())
else:
    final_sample = pd.DataFrame()
    print("Encara no existeix outputmodel_image_sample_4o.csv dins sense-soroll. Primer cal executar la cel.la de models.")

## Resultats generats

Comprovem rapidament que tot ha quedat dins la carpeta separada.

In [ ]:
print("CSV sense-soroll:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges sense-soroll:", len(list(images_dir.glob("*.png"))))
print("Fitxers temporals sense-soroll:", sorted(path.name for path in tmp_dir.glob("*.png")))
print("Logs sense-soroll:", sorted(path.name for path in logs_dir.iterdir()))

## Analisis preliminar de la muestra

Este apartado resume la calidad de las respuestas de los modelos sobre la muestra final. Es un analisis descriptivo previo al analisis estadistico formal: sirve para entender la escala del error, detectar respuestas repetidas o demasiado genericas, y comprobar si el error cambia segun el chroma del color objetivo.

### Variables y formulas usadas

Cada imagen tiene un color objetivo en RGB:

$$
(R_i, G_i, B_i)
$$

Cada modelo devuelve una estimacion, tambien en RGB:

$$
(\hat{R}_{ij}, \hat{G}_{ij}, \hat{B}_{ij})
$$

para la imagen $i$ y el modelo $j$.

El **error cromatico** se calcula como distancia euclidea en el espacio RGB:

$$
error_{ij} = \sqrt{(\hat{R}_{ij} - R_i)^2 + (\hat{G}_{ij} - G_i)^2 + (\hat{B}_{ij} - B_i)^2}
$$

Un error de 0 significa que el modelo ha devuelto exactamente el mismo color RGB que la imagen. Valores mas altos indican mas distancia cromatica.

El **chroma** resume la intensidad o saturacion del color. Lo calculamos convirtiendo RGB a HSV y usando el componente de saturacion $S_i$:

$$
chroma_i = S_i = \frac{\max(R_i, G_i, B_i) - \min(R_i, G_i, B_i)}{\max(R_i, G_i, B_i)}
$$

si $\max(R_i, G_i, B_i) > 0$. Si el maximo es 0, el color es negro puro y definimos $chroma_i = 0$.

Interpretacion:

- chroma cercano a 0: color grisaceo o poco saturado;
- chroma cercano a 1: color muy saturado;
- el chroma no mide brillo, sino separacion relativa entre canales RGB.

### Que miramos aqui

- Distribucion del error por modelo.
- Media, mediana y desviacion tipica del error.
- Grafico violin + boxplot para comparar forma, mediana y dispersion.
- Lazy response: concentracion de respuestas repetidas y uso de colores demasiado genericos.
- Correlacion entre error y chroma.
- Comportamiento por 4 grupos de chroma, usando cuartiles de la muestra.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, display

# Rutas robustas: funciona desde la raiz del repo o desde recollida-dades/notebooks.
_nb_cwd = Path.cwd()
if (_nb_cwd / "recollida-dades").exists():
    _recollida_dir = _nb_cwd / "recollida-dades"
elif _nb_cwd.name == "notebooks" and _nb_cwd.parent.name == "recollida-dades":
    _recollida_dir = _nb_cwd.parent
elif _nb_cwd.name == "recollida-dades":
    _recollida_dir = _nb_cwd
else:
    _recollida_dir = Path("..").resolve()

experiment_dir = _recollida_dir / "experiments" / "sense-soroll"
results_dir = experiment_dir / "results"
plots_dir = experiment_dir / "plots"

final_results_path = results_dir / "sample_colors_solids_tots_models_actual.csv"
error_summary_path = results_dir / "resum_error_solids_tots_models.csv"
lazy_summary_path = results_dir / "resum_lazy_response_solids_tots_models.csv"
accuracy_summary_path = results_dir / "resum_percent_acierto_solids_tots_models.csv"

required_paths = [final_results_path, error_summary_path, lazy_summary_path, accuracy_summary_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Faltan resultados finales: " + ", ".join(str(path) for path in missing_paths))

sample_results = pd.read_csv(final_results_path)
error_summary = pd.read_csv(error_summary_path)
lazy_summary = pd.read_csv(lazy_summary_path)
accuracy_summary = pd.read_csv(accuracy_summary_path)

model_order = ["gpt-4o", "gpt-4o-mini", "gpt-5", "gpt-5.1", "gpt-5.2", "gpt-5.4", "gpt-5.5"]
sample_results = sample_results[sample_results["model"].isin(model_order)].copy()
sample_results["model"] = pd.Categorical(sample_results["model"], categories=model_order, ordered=True)

print("Dataset combinado:", final_results_path)
print("Filas:", len(sample_results))
print("Imagenes unicas:", sample_results["image_name"].nunique())
print("Modelos:", list(sample_results["model"].cat.categories))

display(sample_results.groupby("model", observed=False).size().reset_index(name="n"))


### Descriptiva del error cromatico

Comparamos media, mediana y desviacion tipica para detectar tanto el rendimiento medio como la dispersion de cada modelo.

- La **media** resume el error esperado, pero puede subir mucho si hay casos extremos.
- La **mediana** es mas robusta: indica el error tipico de una respuesta central.
- La **desviacion tipica** mide dispersion: si es alta, el modelo es menos estable entre imagenes.


In [ ]:
# Tabla descriptiva del error cromatico.
error_summary = error_summary.copy()
error_summary["model"] = pd.Categorical(error_summary["model"], categories=model_order, ordered=True)
error_summary = error_summary.sort_values("model")

error_display = error_summary.rename(columns={
    "model": "modelo",
    "n": "n",
    "mean": "media_error",
    "median": "mediana_error",
    "sd": "sd_error",
})
for col in ["media_error", "mediana_error", "sd_error"]:
    error_display[col] = error_display[col].round(2)

display(error_display)

best_mean = error_summary.sort_values("mean").iloc[0]
print(
    f"Menor error medio: {best_mean['model']} "
    f"con media {best_mean['mean']:.2f}."
)


In [ ]:
# Graficos principales del error.
plot_files = [
    "distribucio_error_solids_tots_models.png",
    "violin_boxplot_error_solids_tots_models.png",
]
for name in plot_files:
    path = plots_dir / name
    if path.exists():
        print(path)
        display(DisplayImage(filename=str(path)))
    else:
        print("No encontrado:", path)


### Lazy response

Llamamos **lazy response** a un patron donde el modelo no estima colores concretos, sino que responde con colores muy repetidos o de paleta basica, por ejemplo `00FF00`, `0000FF`, `000000`, `FF00FF` o colores redondeados.

Indicadores usados:

- `unique_hex`: numero de colores distintos que devuelve el modelo;
- `unique_ratio`: proporcion de colores distintos sobre el total de respuestas;
- `max_repeat`: cuantas veces aparece el color mas repetido;
- `top10_repeat_pct`: porcentaje de respuestas concentradas en los 10 colores mas repetidos.

Un modelo con muchas respuestas unicas y bajo `top10_repeat_pct` esta estimando de forma mas fina. Un modelo con pocos colores unicos y alto `top10_repeat_pct` probablemente esta usando respuestas de paleta o aproximaciones demasiado genericas.


In [ ]:
lazy_summary = lazy_summary.copy()
lazy_summary["model"] = pd.Categorical(lazy_summary["model"], categories=model_order, ordered=True)
lazy_summary = lazy_summary.sort_values("model")

lazy_display = lazy_summary.rename(columns={
    "model": "modelo",
    "unique_hex": "colores_unicos",
    "unique_ratio": "ratio_unicos",
    "max_repeat": "max_repeticion",
    "top10_repeat_pct": "top10_repetidos_pct",
})
lazy_display["ratio_unicos"] = lazy_display["ratio_unicos"].round(3)
lazy_display["top10_repetidos_pct"] = lazy_display["top10_repetidos_pct"].round(2)
display(lazy_display)

path = plots_dir / "repeticio_respostes_solids_tots_models.png"
if path.exists():
    print(path)
    display(DisplayImage(filename=str(path)))

worst_lazy = lazy_summary.sort_values("top10_repeat_pct", ascending=False).iloc[0]
print(
    f"Mayor concentracion de respuestas repetidas: {worst_lazy['model']} "
    f"con {worst_lazy['top10_repeat_pct']:.1f}% de respuestas dentro de sus 10 colores mas repetidos."
)


### Relacion entre error y chroma

Aqui comprobamos si los modelos fallan mas en colores poco saturados o muy saturados. Primero calculamos la correlacion entre `error_cromatic` y `chroma` por modelo:

$$
r = corr(error_{ij}, chroma_i)
$$

- Si $r > 0$, el error tiende a aumentar cuando el chroma aumenta.
- Si $r < 0$, el error tiende a disminuir cuando el chroma aumenta.
- Si $r \approx 0$, no hay una relacion lineal clara.

Despues dividimos el chroma en 4 buckets mediante cuartiles. Asi cada grupo contiene aproximadamente el 25% de las imagenes:

- `Q1`: chroma mas bajo;
- `Q2` y `Q3`: chroma intermedio;
- `Q4`: chroma mas alto.

Esto permite ver si un modelo se degrada en una zona concreta de saturacion aunque la correlacion global sea pequena.


In [ ]:
# Correlacion error-chroma y comportamiento por 4 buckets de chroma.
analysis_df = sample_results.dropna(subset=["error_cromatic", "chroma"]).copy()

corr_rows = []
for model, group in analysis_df.groupby("model", observed=False):
    group = group.dropna(subset=["error_cromatic", "chroma"])
    corr = group["error_cromatic"].corr(group["chroma"]) if len(group) > 1 else np.nan
    corr_rows.append({"modelo": str(model), "n": len(group), "corr_error_chroma": corr})

corr_table = pd.DataFrame(corr_rows)
corr_table["corr_error_chroma"] = corr_table["corr_error_chroma"].round(3)
display(corr_table)

# Buckets globales de chroma: se calculan una vez por imagen para que todos los modelos usen los mismos cortes.
image_chroma = analysis_df[["image_name", "chroma"]].drop_duplicates()
image_chroma["chroma_bucket"] = pd.qcut(
    image_chroma["chroma"],
    q=4,
    labels=["Q1 bajo", "Q2 medio-bajo", "Q3 medio-alto", "Q4 alto"],
    duplicates="drop",
)
analysis_df = analysis_df.drop(columns=["chroma_bucket"], errors="ignore").merge(
    image_chroma[["image_name", "chroma_bucket"]],
    on="image_name",
    how="left",
)

bucket_summary = (
    analysis_df.groupby(["model", "chroma_bucket"], observed=False)["error_cromatic"]
    .agg(n="count", media="mean", mediana="median", sd="std")
    .reset_index()
)
bucket_summary["model"] = pd.Categorical(bucket_summary["model"], categories=model_order, ordered=True)
bucket_summary = bucket_summary.sort_values(["model", "chroma_bucket"])
for col in ["media", "mediana", "sd"]:
    bucket_summary[col] = bucket_summary[col].round(2)

display(bucket_summary)

path = plots_dir / "error_vs_chroma_solids_tots_models.png"
if path.exists():
    print(path)
    display(DisplayImage(filename=str(path)))


### Porcentaje de acierto por umbrales

Como el color es continuo, acertar el HEX exacto es una medida demasiado estricta. Por eso usamos varios umbrales de error cromatico:

$$
acierto_t = \frac{1}{n}\sum I(error_{ij} \leq t)
$$

con $t \in \{10, 20, 30, 50\}$. Por ejemplo, `err_le_30_pct` indica el porcentaje de respuestas cuyo error cromatico es como maximo 30 unidades RGB.


In [ ]:
accuracy_summary = accuracy_summary.copy()
accuracy_summary["model"] = pd.Categorical(accuracy_summary["model"], categories=model_order, ordered=True)
accuracy_summary = accuracy_summary.sort_values("model")
accuracy_display = accuracy_summary.rename(columns={
    "model": "modelo",
    "exact_hex_pct": "exacto_hex_pct",
})
for col in accuracy_display.columns:
    if col.endswith("pct"):
        accuracy_display[col] = accuracy_display[col].round(2)
display(accuracy_display)
